# 1차 과제 — 수정본

**실행 안내:** 커널을 재시작하고 모든 셀을 위에서부터 순서대로 실행하세요. 수정 전 출력과 실행 번호를 제거했으며, 수정본의 학습 결과는 아직 생성하지 않았습니다. 데이터와 ImageNet 가중치가 없으면 첫 실행에서 다운로드합니다. 확대된 이미지 배열만 약 4.05 GB이며 학습에는 추가 메모리가 필요합니다.

두 비교 모두 학습 데이터의 마지막 10%를 검증용으로 사용하며, 20 epochs·batch size 100·Adam 조건을 유지합니다. 테스트 데이터는 학습에 사용하지 않습니다. 원래 실험과의 연속성을 위해 모델 구조를 유지하고 조기 종료는 적용하지 않았습니다. 모델별로 필요한 입력 전처리는 다르게 적용합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gc

## 1. Fashion MNIST Sample 그려보기

각 class 별로 한 개 씩의 sample을 뽑아서 그림을 그려봅니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, models, Input
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D

# 재현성을 위해 시드 고정 (환경에 따라 결과는 달라질 수 있음)
tf.keras.utils.set_random_seed(42)
print(tf.__version__)

In [ ]:
# Fashion MNIST 불러오기
fashion_mnist = datasets.fashion_mnist
(train_x, train_y), (test_x, test_y) = fashion_mnist.load_data()
# 학습 및 테스트 이미지 정규화
train_x = train_x.astype('float32') / 255.0
test_x = test_x.astype('float32') / 255.0

# 학습 및 테스트 데이터의 이미지와 정답 레이블 배열 형태(shape) 확인
print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)

In [ ]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

plt.figure(figsize=(5, 2.5))

for class_id in range(10):
    # 해당 클래스에 속하는 첫 번째 이미지의 인덱스
    index = np.where(train_y == class_id)[0][0]

    plt.subplot(2, 5, class_id + 1)
    plt.imshow(train_x[index], cmap=plt.cm.binary)
    plt.title(class_names[class_id], fontsize=8)
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

## 2. Fashion MNIST(or CIFAR10)를 분류하는 CNN 학습하기

CNN이 image data에서 DNN에 비해 얼마나 뛰어난지를 확인하는 것이 목표입니다.  
모수의 수가 비슷한 DNN과 CNN을 구축한 뒤 데이터를 이용하여 각각의 모형을 학습하고 test accuracy를 비교합니다.

In [ ]:
## DNN model을 구축합니다.
dnn_model = models.Sequential([
  Input(shape=(28, 28)),
  Flatten(),
  Dense(110, activation='relu'),
  Dense(100, activation='relu'),
  Dense(10, activation='softmax')]
)

dnn_model.summary()

In [ ]:
# Adam 최적화 알고리즘과 정수 레이블용 다중 분류 손실 함수를 설정하고, 정확도로 성능 평가
dnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습 데이터의 10%를 검증용으로 분리하고, 나머지를 배치 크기 100으로 20회 학습하여 기록 저장
dnn_hist = dnn_model.fit(train_x, train_y, epochs=20, batch_size=100, validation_split=0.10)

In [ ]:
# DNN 모델을 평가합니다.
dnn_sc = dnn_model.evaluate(test_x, test_y)

print('Performance of DNN')
print('...test accuracy: %.3f, test loss: %.3f'%(dnn_sc[1], dnn_sc[0]))

In [ ]:
# CNN 모델에서 사용하기 위해 Data shape을 변경합니다.
ext_train_x = train_x.reshape((60000, 28, 28, 1))
ext_test_x = test_x.reshape((10000, 28, 28, 1))

In [ ]:
## CNN model을 구축합니다.
fashion_cnn_model = models.Sequential([
  Input(shape=(28, 28, 1)),
  Conv2D(32, (3, 3), activation='relu'),
  MaxPool2D(pool_size=(2, 2), strides=(2,2), padding='valid'),
  Conv2D(64, (3, 3), activation='relu'),
  MaxPool2D(pool_size=(2, 2), strides=(2,2), padding='valid'),
  Conv2D(64, (3, 3), activation='relu'),
  Flatten(),
  Dense(64, activation='relu'),
  Dense(10, activation='softmax')
])

fashion_cnn_model.summary()

In [ ]:
# Adam 최적화 알고리즘과 정수 레이블용 다중 분류 손실 함수를 설정하고, 정확도로 성능 평가
fashion_cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습 데이터의 10%를 검증용으로 분리하고, 나머지를 배치 크기 100으로 20회 학습하여 기록 저장
fashion_cnn_hist=fashion_cnn_model.fit(ext_train_x, train_y, epochs=20, batch_size=100, validation_split=0.10)

In [ ]:
# CNN 모델을 평가합니다.
fashion_cnn_sc = fashion_cnn_model.evaluate(ext_test_x, test_y)

print('Performance of CNN')
print('...test accuracy: %.3f, test loss: %.3f'%(fashion_cnn_sc[1], fashion_cnn_sc[0]))

In [ ]:
# DNN, CNN 모델의 결과를 비교합니다.
plt.figure(figsize=(7,5))
plt.plot(dnn_hist.history['accuracy'], 'r--', label='train (DNN)')
plt.plot(dnn_hist.history['val_accuracy'], 'r-', label='valid (DNN)')
plt.plot(fashion_cnn_hist.history['accuracy'], 'b--', label='train (CNN)')
plt.plot(fashion_cnn_hist.history['val_accuracy'], 'b-', label='valid (CNN)')
plt.ylim([0.8,1.005])
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(loc='upper left', fontsize=12)
plt.title('Accuracy trace plot', fontsize=15)
plt.show()

In [ ]:
# 동일한 테스트 데이터에서 모수 수와 성능 비교
print(f"{'Model':<15} {'Parameters':>12} {'Test accuracy':>15} {'Test loss':>12}")
for name, model_obj, score in [('DNN', dnn_model, dnn_sc), ('CNN', fashion_cnn_model, fashion_cnn_sc)]:
    print(f'{name:<15} {model_obj.count_params():>12,} {score[1]:>14.2%} {score[0]:>12.4f}')
print(f'CNN - DNN 테스트 정확도 차이: {(fashion_cnn_sc[1] - dnn_sc[1]) * 100:+.2f}%p')

## 3. 전이학습 활용해보기

전이학습을 실제로 활용해보는 것을 목표로 합니다.  
InceptionV3을 이용하여 CIFAR10의 feature vector를 구축하고  
이 feature vector를 이용하여 multi-class logistic regression model을 학습합니다.  
이 모형의 결과를 앞서 CNN으로 학습한 모형의 결과와 비교해봅니다.

앞서 Fashion MNIST를 DNN, CNN으로 학습했습니다.  
전이학습결과와의 비교를 위해서 CIFAR10을 CNN으로 학습합니다.

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

In [ ]:
# Fashion MNIST와 다른 변수에 이미지와 정답을 함께 저장합니다.
(cifar_train_x, cifar_train_y), (cifar_test_x, cifar_test_y) = datasets.cifar10.load_data()
cifar_train_y = cifar_train_y.reshape(-1)
cifar_test_y = cifar_test_y.reshape(-1)
print(cifar_train_x.shape, cifar_train_y.shape)
print(cifar_test_x.shape, cifar_test_y.shape)

In [ ]:
# 배치별 확대 및 정규화로 큰 임시 배열의 메모리 사용을 줄입니다.
def resize_for_cnn(images, batch_size=256):
    result = np.empty((len(images), 75, 75, 3), dtype=np.float32)
    for start in range(0, len(images), batch_size):
        batch = tf.cast(images[start:start + batch_size], tf.float32)
        result[start:start + batch_size] = tf.image.resize(batch, (75, 75)).numpy() / 255.0
    return result

big_train_x = resize_for_cnn(cifar_train_x)
big_test_x = resize_for_cnn(cifar_test_x)
print(big_train_x.shape, big_test_x.shape)

In [ ]:
# 이미 0~1로 정규화했으므로 다시 255로 나누지 않습니다.
assert len(big_train_x) == len(cifar_train_y)
assert len(big_test_x) == len(cifar_test_y)
assert 0 <= big_train_x.min() <= big_train_x.max() <= 1
assert 0 <= big_test_x.min() <= big_test_x.max() <= 1
print('CNN 입력 범위:', big_train_x.min(), big_train_x.max())

In [ ]:
class_names = [
    'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
    'Dog', 'Frog', 'Horse', 'Ship', 'Truck'
]

labels = np.asarray(cifar_train_y).reshape(-1)
plt.figure(figsize=(5, 2.5))

for class_id in range(10):
    # 해당 클래스에 속하는 첫 번째 이미지의 인덱스
    index = np.where(labels == class_id)[0][0]

    plt.subplot(2, 5, class_id + 1)
    plt.imshow(big_train_x[index])
    plt.title(class_names[class_id], fontsize=8)
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

In [ ]:
## CNN model을 구축합니다.
cifar_cnn_model = models.Sequential([
    Input(shape=(75, 75, 3)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='valid'),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='valid'),
    Conv2D(64, (3, 3), activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

cifar_cnn_model.summary()

In [ ]:
# Adam 최적화 알고리즘과 정수 레이블용 다중 분류 손실 함수를 설정하고, 정확도로 성능 평가
cifar_cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습 데이터의 10%를 검증용으로 분리하고, 나머지를 배치 크기 100으로 20회 학습하여 기록 저장
cifar_cnn_hist=cifar_cnn_model.fit(big_train_x, cifar_train_y, epochs=20, batch_size=100, validation_split=0.10)

In [ ]:
# CNN 모델을 평가합니다.
cifar_cnn_sc = cifar_cnn_model.evaluate(big_test_x, cifar_test_y)

print('Performance of CNN')
print('...test accuracy: %.3f, test loss: %.3f'%(cifar_cnn_sc[1], cifar_cnn_sc[0]))

In [ ]:
# 고정된 사전학습 특징 추출기: 이미지별 2048차원 벡터 출력
inception_model = InceptionV3(
    weights='imagenet', include_top=False,
    input_shape=(75, 75, 3), pooling='avg'
)
inception_model.trainable = False

In [ ]:
# CNN용 0~1 데이터는 유지하고 배치별로 InceptionV3 전처리를 적용합니다.
def extract_features(images, batch_size=100):
    features = np.empty((len(images), 2048), dtype=np.float32)
    for start in range(0, len(images), batch_size):
        # preprocess_input은 0~255를 받아 -1~1로 변환합니다.
        batch = preprocess_input(images[start:start + batch_size] * 255.0)
        features[start:start + batch_size] = inception_model(batch, training=False).numpy()
        if start == 0 or (start + batch_size) % 5000 == 0:
            print(f'특징 추출: {min(start + batch_size, len(images))}/{len(images)}')
    return features

train_features = extract_features(big_train_x)
test_features = extract_features(big_test_x)
print(train_features.shape)  # (50000, 2048)
print(test_features.shape)   # (10000, 2048)

In [ ]:
multi_logistic_model = models.Sequential([
  Input(shape=(2048,)),
  Dense(10, activation='softmax')]
)
multi_logistic_model.summary()

In [ ]:
multi_logistic_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

transfer_hist = multi_logistic_model.fit(train_features, cifar_train_y, epochs=20, batch_size=100, validation_split=0.10)

In [ ]:
# CNN과 동일한 CIFAR-10 테스트 이미지 및 레이블로 평가합니다.
transfer_sc = multi_logistic_model.evaluate(test_features, cifar_test_y)
print(f"{'Model':<22} {'Test accuracy':>15} {'Test loss':>12}")
for name, score in [('CNN', cifar_cnn_sc), ('InceptionV3 + Logistic', transfer_sc)]:
    print(f'{name:<22} {score[1]:>14.2%} {score[0]:>12.4f}')
print(f'전이학습 - CNN 테스트 정확도 차이: {(transfer_sc[1] - cifar_cnn_sc[1]) * 100:+.2f}%p')

In [ ]:
# 정확도와 손실을 함께 확인하여 과적합 여부를 비교합니다.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for history, name, color in [(cifar_cnn_hist, 'CNN', 'b'), (transfer_hist, 'Transfer', 'r')]:
    epochs = np.arange(1, len(history.history['accuracy']) + 1)
    for ax, metric in zip(axes, ['accuracy', 'loss']):
        ax.plot(epochs, history.history[metric], color + '--', label=f'train ({name})')
        ax.plot(epochs, history.history['val_' + metric], color + '-', label=f'valid ({name})')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.capitalize())
        ax.legend(fontsize=8)
axes[0].set_ylim(0, 1.005)
plt.tight_layout()
plt.show()

## 결과 해석

위 표의 **테스트 정확도끼리** 비교합니다. 검증 정확도를 다른 모델의 테스트 정확도와 직접 비교하지 않습니다. 정확도 차이는 이번 한 번의 실행 결과이며 일반적인 우열이나 통계적 유의성을 뜻하지 않습니다.

학습 정확도가 높아져도 검증 정확도가 정체되거나 검증 손실이 증가하면 과적합을 의심할 수 있습니다. 전이학습의 학습 정확도가 CNN보다 낮아도 테스트 정확도가 더 높을 수 있습니다. 학습 정확도만으로 모델의 우열을 판단하지 않습니다.

Fashion MNIST의 DNN은 모수 98,460개, CNN은 93,322개로 규모가 비슷합니다. CIFAR-10의 75×75 CNN은 모수가 978,634개이며 Flatten 다음 Dense(64)에 921,664개가 집중되어 있습니다. 이미지 확대는 새로운 정보를 추가하지 않으면서 이 모델의 모수 수를 늘립니다. 과적합이 관찰되면 후속 실험에서 검증 데이터 기반 조기 종료나 정규화를 검토할 수 있습니다. 테스트 정확도로 epoch를 선택하지 않습니다.

전이학습은 ImageNet 사전학습 특징을 고정하고, 모수 20,490개인 다중 클래스 로지스틱 회귀만 학습합니다. InceptionV3 전체의 모수가 20,490개라는 뜻은 아닙니다. 사전학습 데이터와 모델 규모가 다르므로 결과는 **이번 CNN 구성과 사전학습 특징을 활용한 분류 방식의 비교**로 해석합니다. 시간 비교 시에는 특징 추출 시간도 포함해야 합니다.

실행 후 제출용 해석에는 두 데이터셋 각각의 테스트 정확도와 모델 간 차이, 학습·검증 곡선에서 관찰한 과적합 여부를 실제 출력값에 근거하여 작성합니다. 수정 전 수치는 재사용하지 않습니다.

참고: [InceptionV3 공식 문서](https://keras.io/api/applications/inceptionv3/inception_v3_model/), [전처리 문서](https://keras.io/api/applications/inceptionv3/inception_v3_preprocessing/)